# Drive → SRT (ruso) con `faster-whisper-xxl` (large-v3)

Notebook autocontenido. Levanta videos desde tu Drive (carpeta **Host Videos**), los transcribe a SRT en ruso con el binario standalone **Faster-Whisper-XXL r245.4** de Purfview (large-v3), y guarda los `.srt` en tu Drive (carpeta **Subs_RU**).

**Son 2 celdas:** la primera baja+extrae el binario, monta Drive, lista archivos y te pide el rango a procesar; la segunda transcribe los del rango y guarda. Suena un ruidito cuando termina.

**Antes de correr:** `Runtime → Change runtime type → T4 GPU` (o cualquier GPU con soporte de `float16`).

## Procesamiento en paralelo entre cuentas

Para 500+ videos, abrí 5 cuentas de Colab y repartí el lote por rango. En cada cuenta vas a ver el total y un cuadro para escribir el rango (ej. `1-100`, `101-200`, etc.). El orden es alfabético por nombre, **determinístico**: si las 5 cuentas ven los mismos archivos en `Host Videos`, los rangos no se pisan.

## Parámetros (idénticos a tu llamada de PowerShell)

```
--model large-v3            --language ru
--compute_type float16      --beam_size 10
--temperature 0             --condition_on_previous_text False
--compression_ratio_threshold 2.0
--vad_filter True           --vad_threshold 0.4
--task transcribe           --initial_prompt "Лекция Александра Романова..."
--max_line_width 200        --max_line_count 1   --sentence
```


## 1) Setup + montar Drive + elegir rango

Corré esta celda. Hace:
1. Instala `ffmpeg` y `p7zip`, descarga el binario `Faster-Whisper-XXL r245.4` para Linux (~1.54 GB, una sola vez por sesión) y lo descomprime.
2. Monta tu Drive.
3. Lista los archivos en `MyDrive/Host Videos` y te muestra el **total**.
4. Te muestra un cuadro para que escribas el rango a procesar (ej. `1-100`). Si dejás vacío o ponés `all`, procesa todo.

Si tus carpetas se llaman distinto, editá `INPUT_DIR` / `OUTPUT_DIR` abajo. Extensiones aceptadas: `.mp4 .mkv .webm .mov .avi .m4v .m4a .mp3 .wav .ogg .opus .aac .flac`.


In [ ]:
!apt-get -qq install -y ffmpeg p7zip-full > /dev/null

import os, time, re, subprocess, shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import drive

# --- 1) Descargar + extraer faster-whisper-xxl (una sola vez por sesión) ---
XXL_URL = "https://github.com/Purfview/whisper-standalone-win/releases/download/Faster-Whisper-XXL/Faster-Whisper-XXL_r245.4_linux.7z"
INSTALL_DIR = Path("/content/whisper")
EXE = INSTALL_DIR / "Faster-Whisper-XXL" / "faster-whisper-xxl"
ARCHIVE = INSTALL_DIR / "fwxxl.7z"

if EXE.exists():
    print(f"Binario ya instalado: {EXE}")
else:
    INSTALL_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE.exists():
        print("Descargando Faster-Whisper-XXL r245.4 para Linux (~1.54 GB, una sola vez)...")
        rc = subprocess.call(["wget", "-q", "--show-progress", "-O", str(ARCHIVE), XXL_URL])
        assert rc == 0 and ARCHIVE.stat().st_size > 100 * 1024 * 1024, \
            f"Descarga falló o quedó incompleta (size={ARCHIVE.stat().st_size if ARCHIVE.exists() else 0})"
    print("Extrayendo (puede tardar 1-2 min)...")
    rc = subprocess.call(["7z", "x", "-y", "-bso0", "-bsp0", f"-o{INSTALL_DIR}", str(ARCHIVE)])
    assert rc == 0 and EXE.exists(), f"Extracción falló — no apareció {EXE}"
    ARCHIVE.unlink(missing_ok=True)
os.chmod(EXE, 0o755)

# Verificar versión (sanity check rápido del binario)
try:
    out = subprocess.run([str(EXE), "--help"], capture_output=True, text=True, timeout=30)
    print(f"\n{EXE.name}: OK (--help sale en {len(out.stdout.splitlines())} líneas)")
except Exception as ex:
    print(f"[WARN] el binario no respondió bien a --help: {ex}")

# --- 2) Mount Drive ---
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
print("\nDrive montado.")

# --- 3) Carpetas (editar si tus nombres son distintos) ---
INPUT_DIR  = Path("/content/drive/MyDrive/Host Videos")
OUTPUT_DIR = Path("/content/drive/MyDrive/Subs_RU")

assert INPUT_DIR.exists(), f"No existe la carpeta de entrada: {INPUT_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v",
              ".m4a", ".mp3", ".wav", ".ogg", ".opus", ".aac", ".flac"}
inputs = sorted((p for p in INPUT_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in VIDEO_EXTS),
                key=lambda p: p.name)
assert inputs, f"No hay archivos de video/audio en {INPUT_DIR}"

total = len(inputs)
already = sum(1 for p in inputs if (OUTPUT_DIR / f"{p.stem}-RU.srt").exists())
pending = total - already

print(f"\n📁 {total} archivo(s) en '{INPUT_DIR.name}'  |  ✓ {already} con SRT  |  → {pending} pendientes")
print(f"\nPrimeros 5:")
for i, p in enumerate(inputs[:5], 1):
    mark = "✓" if (OUTPUT_DIR / f"{p.stem}-RU.srt").exists() else "→"
    print(f"  {i:>4}. [{mark}] {p.name}")
if total > 10:
    print(f"  ...")
    for i, p in enumerate(inputs[-3:], total - 2):
        mark = "✓" if (OUTPUT_DIR / f"{p.stem}-RU.srt").exists() else "→"
        print(f"  {i:>4}. [{mark}] {p.name}")

# --- 4) Sugerir reparto entre 5 cuentas ---
N_ACCOUNTS = 5
chunk = (total + N_ACCOUNTS - 1) // N_ACCOUNTS
print(f"\n💡 Reparto sugerido para {N_ACCOUNTS} cuentas (≈{chunk} videos cada una):")
for k in range(N_ACCOUNTS):
    a = k * chunk + 1
    b = min((k + 1) * chunk, total)
    if a > total: break
    print(f"   cuenta {k+1} → {a}-{b}")

# --- 5) Cuadro de input para el rango ---
range_box = widgets.Text(
    value=f"1-{min(100, total)}",
    placeholder="ej: 1-100  (o 'all' para todo)",
    description="Rango:",
    layout=widgets.Layout(width="60%"),
    style={"description_width": "60px"},
)
print(f"\nElegí el rango (1..{total}) y pasá al paso 2:")
display(range_box)


## 2) Transcribir el rango → SRT en Drive

Una sola celda hace el resto:

1. Lee el rango del cuadro de arriba (`1-100`, `1`, `all`, …).
2. Por cada video del rango llama al binario `faster-whisper-xxl` con los flags exactos de tu PowerShell — incluyendo `--sentence`, `--max_line_width 200`, `--max_line_count 1`.
3. El CLI escribe el SRT en una carpeta temporal `/content/srt_tmp/` y después se mueve a `MyDrive/Subs_RU/<stem>-RU.srt` (más rápido y robusto que escribir directo en Drive).
4. Si un SRT ya estaba en Drive, se saltea.


In [ ]:
import numpy as np
from IPython.display import Audio, display

INITIAL_PROMPT = (
    "Лекция Александра Романова о бестопливных генераторах. Термины: ПН-переход, разрядник, качер, варикап, тиристор, лавинный диод, туннельный диод, Тесла, фузьки, бифилярная катушка."
)

# --- 1) Parsear el rango del cuadro del paso 1 ---
assert "range_box" in globals(), "Primero corré el paso 1."
spec = (range_box.value or "").strip().lower()
if spec in ("", "all"):
    start, end = 1, total
else:
    m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", spec) or re.fullmatch(r"(\d+)", spec)
    assert m, f"Rango inválido: {spec!r}. Usá '1-100' o '50' o 'all'."
    if m.lastindex == 1:
        start = end = int(m.group(1))
    else:
        start, end = int(m.group(1)), int(m.group(2))
start = max(1, start); end = min(total, end)
assert start <= end, f"Rango vacío después de recortar a 1..{total}: {start}-{end}"

batch = inputs[start-1:end]
print(f"Procesando {len(batch)} archivo(s): #{start} a #{end} de {total}.")

# --- 2) Carpeta tmp donde el CLI deja los SRT antes de moverlos a Drive ---
TMP_OUT = Path("/content/srt_tmp")
TMP_OUT.mkdir(exist_ok=True)

done = skipped = failed = 0
t_global = time.time()

for offset, vid in enumerate(batch):
    i = start + offset
    final_srt = OUTPUT_DIR / f"{vid.stem}-RU.srt"
    if final_srt.exists():
        print(f"\n[{i}/{end}] SALTADO (ya existe en Drive): {final_srt.name}")
        skipped += 1
        continue
    # limpiar cualquier SRT viejo en tmp para este stem
    for f in TMP_OUT.glob(f"{vid.stem}.*"):
        try: f.unlink()
        except: pass
    print(f"\n[{i}/{end}] Procesando: {vid.name}")
    t0 = time.time()
    cmd = [
        str(EXE),
        str(vid),
        "--model", "large-v3",
        "--language", "ru",
        "--task", "transcribe",
        "--compute_type", "float16",
        "--beam_size", "10",
        "--temperature", "0",
        "--condition_on_previous_text", "False",
        "--compression_ratio_threshold", "2.0",
        "--vad_filter", "True",
        "--vad_threshold", "0.4",
        "--initial_prompt", INITIAL_PROMPT,
        "--max_line_width", "200",
        "--max_line_count", "1",
        "--sentence",
        "--output_dir", str(TMP_OUT),
        "--output_format", "srt",
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        cli_srt = TMP_OUT / f"{vid.stem}.srt"
        if result.returncode == 0 and cli_srt.exists():
            shutil.move(str(cli_srt), str(final_srt))
            n_cues = sum(1 for L in final_srt.read_text(encoding="utf-8").splitlines()
                         if L.strip().isdigit())
            print(f"   ✓ {n_cues} cues, {time.time()-t0:.1f}s → {final_srt.name}")
            done += 1
        else:
            tail_out = (result.stdout or "")[-400:]
            tail_err = (result.stderr or "")[-400:]
            print(f"   ✗ CLI exit {result.returncode}; sin SRT.")
            if tail_out: print(f"   stdout (últimas 400 chars):\n{tail_out}")
            if tail_err: print(f"   stderr (últimas 400 chars):\n{tail_err}")
            failed += 1
    except Exception as ex:
        print(f"   ✗ excepción: {ex}")
        failed += 1

print(f"\n=== Resumen del rango {start}-{end} ===")
print(f"  procesados: {done}")
print(f"  saltados:   {skipped}")
print(f"  fallidos:   {failed}")
print(f"  tiempo total: {(time.time()-t_global)/60:.1f} min")
print(f"\nSRT en: {OUTPUT_DIR}")

# Ruidito final
sr = 22050
out_audio = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    env = np.exp(-3*t)
    out_audio = np.concatenate([out_audio, (0.3*env*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out_audio, rate=sr, autoplay=True))
